# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa - Exploration with `mlcroissant`

This notebook provides an interactive guide for loading and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema, available at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

This dataset contains survey data collected from residents of Kilifi County, focusing on mental health indicators, alongside demographic details and standardized questionnaire results (e.g., GAD-7, PHQ-9, PSQ).

In [ ]:
# Ensure 'mlcroissant' is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
# The metadata exposes dataset details
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Dataset version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Date published: {metadata.datePublished}")

## 2. Data Overview
Discover available record sets, fields, and columns in this dataset. All entities are referenced by their `@id`.

In [ ]:
# List record sets and their details by @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  Name: {rs['name'] if 'name' in rs else ''}")
    print(f"  Description: {rs.get('description', '')}")
    # Show available fields for each record set
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            field_obj = dataset.field_by_id(field['@id']) if isinstance(field, dict) else dataset.field_by_id(field)
            print(f"    - {field_obj['@id']}: {field_obj['name']}")

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis.

**Note:** All data extractions use `@id` references for record sets and fields for precise control.

In [ ]:
# You can change these IDs to any record set listed above
# Here we look for the main survey records.
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/3b2c271b-f7e4-4ace-adf6-ccf8c10c351d'  # example placeholder, adjust as per overview

# If unsure, you can set this to the first listed record set
# List all record set ids
all_record_set_ids = [rs['@id'] for rs in record_sets]

dfs = {}

for rs_id in all_record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    # For large datasets, you can slice or sample instead of list(...)
    try:
        rows = list(dataset.records(record_set=rs_id))
        dfs[rs_id] = pd.DataFrame(rows)
        print(f"  --> Columns: {dfs[rs_id].columns.tolist()}")
        print(f"  --> {len(dfs[rs_id])} records loaded")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# View first few rows of the main record set
main_df = dfs[main_record_set_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's look at numeric fields and do basic processing—such as filtering, normalizing, and grouping—using fields referenced by their `@id`.

In [ ]:
# Suppose one field is 'gad7_score' referenced by @id
# Use the field @id directly from the Croissant schema
numeric_field_id = None
# Find a suitable numeric field
for col in main_df.columns:
    if 'gad7' in col.lower() or 'score' in col.lower() or 'phq9' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field such as GAD-7 or PHQ-9 found in columns.")
else:
    print(f"Selected numeric field: {numeric_field_id}")

    # Show distribution of this field
    print(main_df[[numeric_field_id]].describe())

    # Filter records for a threshold
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field, e.g. gender (by @id)
    group_field_id = None
    for col in main_df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)
    else:
        print("No grouping field (e.g., gender) found.")

## 5. Visualization

Visualize distribution of the selected numeric field and its grouping (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped boxplot (if group_field_id is present)
    if group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df.dropna(subset=[group_field_id, numeric_field_id]))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we successfully loaded the FAIR² Mental Health Survey Dataset schema using `mlcroissant`, inspected its record sets, and explored survey results—including mental health scores—by referencing all data entities through their Croissant `@id` fields.

- The dataset contains rich demographic and psychological symptom data for Kilifi County, Kenya.
- Filtering and visualization by specific metrics (such as GAD-7, PHQ-9, or PSQ) and demographic categories reveals trends and differences in mental health indicators, which can inform public health analyses.

Feel free to extend this notebook by referencing additional record sets or fields by their `@id` for deeper domain exploration!